In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

ticker = "İran"  # arama terimi olarak kullanılacak

# Google News RSS - tamamen ücretsiz, API yok
url = f"https://news.google.com/rss/search?q={ticker}&hl=en&gl=US&ceid=US:en"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)

parser = BeautifulSoup(response.text, "xml")
items = parser.find_all("item")

sia = SentimentIntensityAnalyzer()
sentiments = []

for item in items:
    title = item.find("title").text
    pub_date = item.find("pubDate").text[:16]  # tarih kısmı
    source = item.find("source").text if item.find("source") else "unknown"
    link = item.find("link").text

    score = sia.polarity_scores(title)

    sentiments.append({
        "ticker": ticker,
        "date": pub_date,
        "source": source,       # hangi haber sitesi
        "title": title,
        "compound": score["compound"],
        "positive": score["pos"],
        "negative": score["neg"],
        "neutral":  score["neu"]
    })

df = pd.DataFrame(sentiments)
df = df.set_index("date")

print(df[["ticker", "source", "title", "compound"]].to_string())

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


                 ticker                          source                                                                                                                                                                       title  compound
date                                                                                                                                                                                                                                         
Fri, 22 May 2026   İran                        CBS News                                                    Live Updates: Rubio says "slight progress" in Iran peace talks, but rejects Strait of Hormuz "tolling system" - CBS News   -0.2846
Fri, 22 May 2026   İran                      Al Jazeera                                                                                      Iran war live: ‘Little bit’ of progress in US-Iran talks, Washington says - Al Jazeera   -0.2732
Fri, 22 May 2026   İran                         

In [9]:
df

,ticker,source,title,compound,positive,negative,neutral
date,,,,,,,
"Fri, 22 May 2026",İran,CBS News,"Live Updates: Rubio says ""slight progress"" in ...",-0.2846,0.170,0.176,0.654
"Fri, 22 May 2026",İran,Al Jazeera,Iran war live: ‘Little bit’ of progress in US-...,-0.2732,0.150,0.209,0.642
"Fri, 22 May 2026",İran,CNN,Abu Dhabi is ‘doubling down’ on tourism despit...,0.4847,0.239,0.000,0.761
"Fri, 22 May 2026",İran,Axios,Pakistani field marshal heads to Tehran to try...,0.0000,0.000,0.000,1.000
"Fri, 22 May 2026",İran,Bloomberg.com,India Central Bank Says Near Term Outlook Clou...,-0.6249,0.000,0.338,0.662
...,...,...,...,...,...,...,...
"Fri, 22 May 2026",İran,WVNS,Oil prices could spike next week without Iran ...,0.0000,0.000,0.000,1.000
"Thu, 21 May 2026",İran,Financial Times,Marco Rubio sees ‘good signs’ US could reach d...,0.0258,0.084,0.000,0.916
"Thu, 21 May 2026",İran,Reuters,Exclusive: Supreme Leader says enriched uraniu...,0.6249,0.298,0.000,0.702


In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
compound,100.0,-0.172401,0.435092,-0.8689,-0.5994,-0.08945,0.00645,0.8442
positive,100.0,0.073560,0.110643,0.0000,0.0000,0.00000,0.15075,0.5030
negative,100.0,0.143480,0.141206,0.0000,0.0000,0.16650,0.25325,0.5600
neutral,100.0,0.782980,0.152437,0.4400,0.6820,0.76600,0.91075,1.0000
